# Notebook 4: Check Training Progress

**For beginners.** This notebook helps you answer:
- Is my GPU working?
- How much memory is being used?
- Is training still running?
- What was the last loss value?

**Use this anytime** — while training runs or after it finishes.


## Check 1: Is My GPU Working?

Run this to see your GPU name, memory, and if PyTorch can see it.

In [ ]:
import torch

print("=" * 50)
print("GPU STATUS CHECK")
print("=" * 50)

if not torch.cuda.is_available():
    print("❌ CUDA is NOT available!")
    print("   Training will use CPU (very slow).")
else:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_gb = props.total_memory / (1024**3)
        reserved_gb = torch.cuda.memory_reserved(i) / (1024**3)
        allocated_gb = torch.cuda.memory_allocated(i) / (1024**3)
        free_gb = total_gb - reserved_gb
        
        print(f"\n🎮 GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"   Total memory:     {total_gb:.1f} GB")
        print(f"   Reserved:         {reserved_gb:.1f} GB")
        print(f"   Allocated:        {allocated_gb:.1f} GB")
        print(f"   Free:             {free_gb:.1f} GB")
        print(f"   Usage:            {(allocated_gb/total_gb)*100:.1f}%")
        
        if allocated_gb > 1:
            print("   ✅ Something is using the GPU!")
        else:
            print("   ⚠️ GPU is idle (no training running)")

## Check 2: Check Training Logs

Training saves logs to `artifacts/logs/`. Let's see the latest log file.

In [ ]:
import os
import glob

log_dir = '../artifacts/logs'

print("=" * 50)
print("TRAINING LOGS")
print("=" * 50)

if not os.path.exists(log_dir):
    print("❌ No logs folder found.")
    print("   Training may not have started yet.")
else:
    log_files = glob.glob(os.path.join(log_dir, '*.log'))
    
    if not log_files:
        print("⚠️ No log files yet.")
        print("   Wait a few minutes for training to start.")
    else:
        # Get the most recent log file
        latest_log = max(log_files, key=os.path.getmtime)
        print(f"📄 Latest log: {os.path.basename(latest_log)}\n")
        
        # Show last 20 lines
        with open(latest_log, 'r') as f:
            lines = f.readlines()
            last_lines = lines[-20:]
            print("Last 20 lines:")
            print("-" * 50)
            for line in last_lines:
                print(line.rstrip())
            print("-" * 50)
            
        # Count total lines as rough progress indicator
        print(f"\n📊 Total log lines: {len(lines)}")

## Check 3: Check Checkpoints

Checkpoints are saved models during training. If training crashes, you can resume from the last checkpoint.

In [ ]:
checkpoint_dir = '../artifacts/checkpoints'

print("=" * 50)
print("CHECKPOINTS")
print("=" * 50)

if not os.path.exists(checkpoint_dir):
    print("❌ No checkpoint folder.")
else:
    # Find checkpoint folders
    checkpoints = [d for d in os.listdir(checkpoint_dir) 
                   if os.path.isdir(os.path.join(checkpoint_dir, d)) and 'checkpoint' in d]
    
    if not checkpoints:
        print("⏳ No checkpoints saved yet.")
        print("   Checkpoints save every 50 steps by default.")
    else:
        checkpoints.sort()
        print(f"✅ Found {len(checkpoints)} checkpoint(s):\n")
        for cp in checkpoints:
            cp_path = os.path.join(checkpoint_dir, cp)
            # Get folder size
            size = sum(os.path.getsize(os.path.join(dirpath, filename)) 
                      for dirpath, dirnames, filenames in os.walk(cp_path) 
                      for filename in filenames)
            size_mb = size / (1024 * 1024)
            print(f"   📁 {cp} ({size_mb:.1f} MB)")
        
        print("\n💡 To resume training from the last checkpoint:")
        print(f"   python scripts/train.py --config configs/default.yaml")

## Check 4: Check Best Model

After training finishes, the best model is saved here. Let's check if it exists.

In [ ]:
best_model_dir = '../artifacts/best_model/final'

print("=" * 50)
print("BEST MODEL")
print("=" * 50)

if not os.path.exists(best_model_dir):
    print("❌ No best model saved yet.")
    print("   Training may still be running.")
else:
    files = os.listdir(best_model_dir)
    print(f"✅ Best model folder exists!\n")
    print(f"   Files ({len(files)} total):")
    for f in sorted(files)[:10]:  # Show first 10
        print(f"      {f}")
    if len(files) > 10:
        print(f"      ... and {len(files)-10} more")
    
    # Check total size
    size = sum(os.path.getsize(os.path.join(best_model_dir, f)) 
              for f in files if os.path.isfile(os.path.join(best_model_dir, f)))
    for root, dirs, filenames in os.walk(best_model_dir):
        for f in filenames:
            fp = os.path.join(root, f)
            size += os.path.getsize(fp)
    size_mb = size / (1024 * 1024)
    print(f"\n   Total size: {size_mb:.1f} MB")
    print("\n💡 You can now use this model for inference!")

## Check 5: Quick Test Your Model

If the best model exists, let's ask it a simple question!

In [ ]:
import sys
sys.path.insert(0, '..')

if not os.path.exists(best_model_dir):
    print("❌ No model to test yet. Run training first!")
else:
    print("🧪 Loading model for a quick test...\n")
    
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained(best_model_dir)
    model = AutoModelForCausalLM.from_pretrained(
        best_model_dir,
        device_map='auto',
        torch_dtype='auto'
    )
    
    # Simple question
    question = "What is machine learning?"
    print(f"Question: {question}\n")
    
    inputs = tokenizer(question, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7
    )
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Answer: {answer}")

## Check 6: Monitor GPU in Real-Time

Run this cell to watch your GPU usage update every 2 seconds.

**Press the STOP button (⏹️) to stop monitoring.**

In [ ]:
import time
from IPython.display import clear_output

print("Starting GPU monitor (updates every 2 seconds)...")
print("Press STOP button to exit\n")

try:
    while True:
        clear_output(wait=True)
        print("=" * 50)
        print("LIVE GPU MONITOR")
        print("=" * 50)
        
        for i in range(torch.cuda.device_count()):
            allocated = torch.cuda.memory_allocated(i) / (1024**3)
            reserved = torch.cuda.memory_reserved(i) / (1024**3)
            total = torch.cuda.get_device_properties(i).total_memory / (1024**3)
            
            print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
            print(f"  Memory: {allocated:.1f} / {total:.1f} GB ({(allocated/total)*100:.0f}%)")
            
            # Simple bar chart
            bar_len = 20
            filled = int((allocated / total) * bar_len)
            bar = '█' * filled + '░' * (bar_len - filled)
            print(f"  [{bar}]")
        
        print("\n(Updating every 2 seconds... Press STOP to exit)")
        time.sleep(2)
        
except KeyboardInterrupt:
    print("\n\nMonitor stopped.")

## Summary

| Check | What It Tells You |
|-------|-------------------|
| GPU Status | Is PyTorch using your GPU? |
| Training Logs | Is training running? What is the loss? |
| Checkpoints | Can you resume if training crashes? |
| Best Model | Is the final model ready? |
| Quick Test | Does the model answer questions? |
| Live Monitor | Real-time GPU usage while training |

**Run this notebook anytime you want to check what's happening!** 🎯
